In [2]:
import pandas as pd

rain = pd.read_csv('/Users/judetak/Desktop/산공카르텔경진대회/서울시 강수량 현황 정보.csv',encoding='cp949')
time = pd.read_csv('/Users/judetak/Desktop/산공카르텔경진대회/unique_times_2023-10_2025-10.csv')

In [7]:
rain['조사년월일']

0         20251229
1         20251229
2         20251229
3         20251229
4         20251229
            ...   
108438    20120102
108439    20120102
108440    20120102
108441    20120102
108442    20120102
Name: 조사년월일, Length: 108443, dtype: int64

In [8]:

# ---------------------------------------------------------
# 1. 측정지역이 '중구구' 또는 '남산구'인 행 모두 제거
# ---------------------------------------------------------
rain = rain[~rain['측정지역'].isin(['중구구', '남산구'])]

# ---------------------------------------------------------
# 2. 2023-10-01 ~ 2025-10-31 날짜와 모든 unique한 측정지역 조합 생성
# ---------------------------------------------------------
# 원하는 기간의 날짜 생성 (YYYY-MM-DD 형식)
dates = pd.date_range(start='2023-10-01', end='2025-10-31')

# 기존 '조사년월일' 데이터와 동일하게 8자리 정수(YYYYMMDD) 형태로 변환
date_ints = dates.strftime('%Y%m%d').astype(int)

# 현재 데이터에 있는 고유한 측정지역 목록 추출
unique_regions = rain['측정지역'].unique()

# 날짜와 측정지역의 모든 가능한 조합(Cartesian Product)을 가진 뼈대 데이터프레임 생성
full_combinations = pd.MultiIndex.from_product(
    [date_ints, unique_regions], 
    names=['조사년월일', '측정지역']
).to_frame(index=False)

# 뼈대 데이터프레임에 기존 rain 데이터를 왼쪽 병합(Left Join)
# - 원래 존재하던 데이터의 강수량, 측정소이름 등은 그대로 유지됨
# - 원래 없었던 날짜+지역 조합은 생성되고, 나머지 값(일일강수량 등)은 NaN이 됨
rain_expanded = pd.merge(full_combinations, rain, on=['조사년월일', '측정지역'], how='left')

# 결과 확인
print("확장된 데이터 크기:", rain_expanded.shape)
print(rain_expanded.head())

확장된 데이터 크기: (21712, 4)
      조사년월일 측정지역 측정소이름  일일강수량
0  20231001  도봉구    도봉    0.0
1  20231001  금천구    금천    0.0
2  20231001  강동구    강동    0.0
3  20231001  강남구    강남    0.0
4  20231001  관악구    관악    0.0


In [10]:
time

,측정일시,kma_tm,date,hour
0,2023-10-04 10:00:00,202310041000,2023-10-04,10
1,2023-10-04 11:00:00,202310041100,2023-10-04,11
2,2023-10-04 12:00:00,202310041200,2023-10-04,12
3,2023-10-04 13:00:00,202310041300,2023-10-04,13
4,2023-10-04 14:00:00,202310041400,2023-10-04,14
...,...,...,...,...
1255,2025-10-29 10:00:00,202510291000,2025-10-29,10
1256,2025-10-29 11:00:00,202510291100,2025-10-29,11
1257,2025-10-29 12:00:00,202510291200,2025-10-29,12
1258,2025-10-29 13:00:00,202510291300,2025-10-29,13


In [11]:
# '-' 기호를 없애고 정수형(int)으로 변환하여 다시 date 컬럼에 저장
time['date'] = time['date'].str.replace('-', '').astype(int)

# 결과 확인
print(time['date'].head())

0    20231004
1    20231004
2    20231004
3    20231004
4    20231004
Name: date, dtype: int64


In [12]:
# 1. time 데이터의 'date' 컬럼에서 고유한(unique) 날짜 값들만 추출
unique_dates = time['date'].unique()

# 2. rain 데이터의 '조사년월일'이 unique_dates 목록에 포함되는 행만 추출
rain_filtered = rain[rain['조사년월일'].isin(unique_dates)]

# 결과 확인
print("필터링된 rain 데이터 크기:", rain_filtered.shape)
print(rain_filtered.head())

필터링된 rain 데이터 크기: (5335, 4)
        조사년월일 측정소이름  측정지역  일일강수량
291  20251017    구로   구로구   13.5
292  20251017    중구    중구   11.0
293  20251017    은평   은평구   12.0
294  20251017    용산   용산구   13.0
295  20251017   영등포  영등포구   10.5


In [14]:
import geopandas as gpd

grid = gpd.read_file('/Users/judetak/Desktop/산공카르텔경진대회/격자_250m_4326.gpkg')

In [15]:
grid

,CELL_ID,CELL_X,CELL_Y,GID,lon,lat,geometry
0,다사54504050,954625,1940625,다사54ba40ba,126.985479,37.464854,"MULTIPOLYGON (((126.98549 37.46259, 126.98548 ..."
1,다사54255675,954375,1956875,다사54ab56bb,126.981639,37.611306,"MULTIPOLYGON (((126.98166 37.60904, 126.98164 ..."
2,다사54255700,954375,1957125,다사54ab57aa,126.981624,37.613560,"MULTIPOLYGON (((126.98164 37.61129, 126.98162 ..."
3,다사54255725,954375,1957375,다사54ab57ab,126.981608,37.615813,"MULTIPOLYGON (((126.98162 37.61355, 126.98161 ..."
4,다사54255750,954375,1957625,다사54ab57ba,126.981593,37.618066,"MULTIPOLYGON (((126.98161 37.6158, 126.98159 3..."
...,...,...,...,...,...,...,...
10120,다사54504125,954625,1941375,다사54ba41ab,126.985433,37.471614,"MULTIPOLYGON (((126.98545 37.46935, 126.98543 ..."
10121,다사54504150,954625,1941625,다사54ba41ba,126.985417,37.473867,"MULTIPOLYGON (((126.98543 37.4716, 126.98542 3..."
10122,다사54503975,954625,1939875,다사54ba39bb,126.985526,37.458094,"MULTIPOLYGON (((126.98554 37.45583, 126.98553 ..."
10123,다사54504000,954625,1940125,다사54ba40aa,126.985510,37.460347,"MULTIPOLYGON (((126.98553 37.45808, 126.98551 ..."


In [17]:
gu = gpd.read_file('/Users/judetak/Downloads/tb_etl_tp_admstr_zone_lgldong_bndry/admstr_zone_lgldong_bndry_24.shp', encoding = 'cp949')

In [19]:


# 1. 서울시 25개 자치구 코드 매핑 사전 (코드 앞 5자리 기준)
seoul_sgg_dict = {
    '11110': '종로구', '11140': '중구', '11170': '용산구', '11200': '성동구', '11215': '광진구',
    '11230': '동대문구', '11260': '중랑구', '11290': '성북구', '11305': '강북구', '11320': '도봉구',
    '11350': '노원구', '11380': '은평구', '11410': '서대문구', '11440': '마포구', '11470': '양천구',
    '11500': '강서구', '11530': '구로구', '11545': '금천구', '11560': '영등포구', '11590': '동작구',
    '11620': '관악구', '11650': '서초구', '11680': '강남구', '11710': '송파구', '11740': '강동구'
}

# 2. EMD_CD를 문자열로 변환 후, 앞 5자리를 추출하여 '자치구' 컬럼 새로 생성
gu['자치구'] = gu['EMD_CD'].astype(str).str[:5].map(seoul_sgg_dict)

# 매핑 결과 확인 (동 이름 옆에 정확한 구 이름이 붙었는지 확인해보세요)
print("--- 매핑 결과 확인 ---")
print(gu[['EMD_CD', 'EMD_NM', '자치구']].head())



--- 매핑 결과 확인 ---
     EMD_CD EMD_NM  자치구
0  11110101    청운동  종로구
1  11110102    신교동  종로구
2  11110103    궁정동  종로구
3  11110104    효자동  종로구
4  11110105    창성동  종로구


In [20]:

# ---------------------------------------------------------
# 3. 격자(grid) 데이터와 공간 결합 및 강수량 데이터 병합
# ---------------------------------------------------------

# 좌표계 통일 (grid와 동일하게 WGS84로 맞춤)
gu = gu.to_crs(epsg=4326)

# [공간 결합] 격자가 어느 동/자치구에 속하는지 매핑
grid_with_gu = gpd.sjoin(grid, gu, how='left', predicate='intersects')

# [속성 결합] 방금 만든 '자치구' 컬럼을 기준으로 rain_filtered 병합
final_data = pd.merge(
    grid_with_gu, 
    rain_filtered, 
    left_on='자치구', 
    right_on='측정지역', 
    how='left'
)

print("\n--- 최종 데이터 확인 ---")
print(final_data.head())


--- 최종 데이터 확인 ---
      CELL_ID  CELL_X   CELL_Y         GID         lon        lat  \
0  다사54504050  954625  1940625  다사54ba40ba  126.985479  37.464854   
1  다사54504050  954625  1940625  다사54ba40ba  126.985479  37.464854   
2  다사54504050  954625  1940625  다사54ba40ba  126.985479  37.464854   
3  다사54504050  954625  1940625  다사54ba40ba  126.985479  37.464854   
4  다사54504050  954625  1940625  다사54ba40ba  126.985479  37.464854   

                                            geometry  index_right    EMD_CD  \
0  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...          420  11620103   
1  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...          420  11620103   
2  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...          420  11620103   
3  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...          420  11620103   
4  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...          420  11620103   

  COL_ADM_SE EMD_NM  SGG_OID  자치구       조사년월일 측정소이름 측정지역  일일강수량  
0      11620    남현동      

In [21]:
final_data = final_data.drop(columns = ['GID','index_right','SGG_OID'])

In [22]:
# '일일강수량' 컬럼의 NaN 값을 0으로 채우기
final_data['일일강수량'] = final_data['일일강수량'].fillna(0)

# 결측치가 잘 채워졌는지 확인 (결과가 0이 나와야 정상입니다)
print("남은 결측치 개수:", final_data['일일강수량'].isna().sum())

# 데이터 확인
print(final_data.head())

남은 결측치 개수: 0
      CELL_ID  CELL_X   CELL_Y         lon        lat  \
0  다사54504050  954625  1940625  126.985479  37.464854   
1  다사54504050  954625  1940625  126.985479  37.464854   
2  다사54504050  954625  1940625  126.985479  37.464854   
3  다사54504050  954625  1940625  126.985479  37.464854   
4  다사54504050  954625  1940625  126.985479  37.464854   

                                            geometry    EMD_CD COL_ADM_SE  \
0  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...  11620103      11620   
1  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...  11620103      11620   
2  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...  11620103      11620   
3  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...  11620103      11620   
4  MULTIPOLYGON (((126.98549 37.46259, 126.98548 ...  11620103      11620   

  EMD_NM  자치구       조사년월일 측정소이름 측정지역  일일강수량  
0    남현동  관악구  20251017.0    관악  관악구    8.5  
1    남현동  관악구  20251016.0    관악  관악구    5.0  
2    남현동  관악구  20251015.0    관악  관악구    4.5

In [24]:
# 결과를 저장할 경로 지정 (원하시는 경로/파일명으로 수정 가능)
save_path = '/Users/judetak/Desktop/산공카르텔경진대회/final_data.parquet'

# Parquet 형식으로 저장 (인덱스는 제외하고 저장)
final_data.to_parquet(save_path, index=False)

print(f"데이터가 성공적으로 저장되었습니다: {save_path}")

데이터가 성공적으로 저장되었습니다: /Users/judetak/Desktop/산공카르텔경진대회/final_data.parquet
